# Zeroshot

### owlvit-base-patch32 

In [0]:
from PIL import Image
import torch
from transformers import OwlViTProcessor, OwlViTForObjectDetection
import matplotlib.pyplot as plt
import matplotlib.patches as patches



def detect_object(img_path, object_to_find, threshold=0.3):
    # Load pre-trained model and processor
    processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
    model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32")

    # Open and process the image
    image = Image.open(img_path)
    texts = [[f"a photo of a {object_to_find}"]]
    inputs = processor(text=texts, images=image, return_tensors="pt")
    outputs = model(**inputs)

    # Process the output
    target_sizes = torch.Tensor([image.size[::-1]])
    results = processor.post_process_object_detection(outputs=outputs, threshold=threshold, target_sizes=target_sizes)

    # Extract results for the first (and only) image
    boxes, scores, labels = results[0]["boxes"], results[0]["scores"], results[0]["labels"]

    # Prepare lists to store all detections
    all_coordinates = []
    all_confidences = []

    # Process all detections
    for box, score in zip(boxes, scores):
        coordinates = [round(i, 2) for i in box.tolist()]
        confidence = round(score.item(), 3)
        all_coordinates.append(coordinates)
        all_confidences.append(confidence)

    # Return all results
    return all_coordinates, all_confidences

def display_detection(img_path, object_name, all_coordinates, all_confidences):
    print(f"Received coordinates: {all_coordinates}")
    try:
        # Load the image
        image = Image.open(img_path)
        
        # Create a figure and axis
        fig, ax = plt.subplots(1)
        
        # Display the image
        ax.imshow(image)

        # Define a color map for different confidence levels
        colors = ['r', 'g', 'b', 'y', 'c', 'm']

        # Create Rectangle patches for each detection
        for coordinates, confidence in zip(all_coordinates, all_confidences):
            color_index = min(int(confidence * len(colors)), len(colors) - 1)
            color = colors[color_index]
            
            # Unpack coordinates
            x1, y1, x2, y2 = coordinates
            
            rect = patches.Rectangle((x1, y1), 
                                     x2 - x1, 
                                     y2 - y1, 
                                     linewidth=2, edgecolor=color, facecolor='none')
            
            # Add the patch to the Axes
            ax.add_patch(rect)

            # Add label
            plt.text(x1, y1 - 10, 
                     f"{object_name} ({confidence:.3f})", 
                     color=color, fontsize=10, weight='bold')

        # Turn off axis
        plt.axis('off')
        
        # Set the title
        plt.title(f"Detected {object_name}s", fontsize=14, fontweight='bold')
        
        # Display the image with the bounding boxes
        plt.show()
    except Exception as e:
        print(f"Error in display_detection: {e}")



file_paths = [i.path.replace("dbfs:","") for i in dbutils.fs.ls("/Volumes/ericos_catalog/default/data/images/image_detection/packages/")]

for img_path in file_paths:

  object_to_find = "package"

  # Get detections (assuming this function works correctly)
  all_coordinates, all_confidences = detect_object(img_path, object_to_find, threshold=0.1)

  # Display detections
  if all_coordinates:
      display_detection(img_path, object_to_find, all_coordinates, all_confidences)
  else:
      print(f"No {object_to_find} detected in the image.")


In [0]:
file_paths = [i.path.replace("dbfs:","") for i in dbutils.fs.ls("/Volumes/ericos_catalog/default/data/images/image_detection/packages/")]

for img_path in file_paths:

  object_to_find = "package"

  # Get detections (assuming this function works correctly)
  all_coordinates, all_confidences = detect_object(img_path, object_to_find = "package", threshold=0.1)

  # Display detections
  if all_coordinates:
      display_detection(img_path, object_to_find, all_coordinates, all_confidences)
  else:
      print(f"No {object_to_find} detected in the image.")


In [0]:
file_paths = [i.path.replace("dbfs:","") for i in dbutils.fs.ls("/Volumes/ericos_catalog/default/data/images/image_detection/signature/")]

for img_path in file_paths:

  object_to_find = "signature"

  # Get detections (assuming this function works correctly)
  all_coordinates, all_confidences = detect_object(img_path, object_to_find = "signature", threshold=0.033)

  # Display detections
  if all_coordinates:
      display_detection(img_path, object_to_find, all_coordinates, all_confidences)
  else:
      image = Image.open(img_path)
      fig, ax = plt.subplots(1)
      ax.imshow(image)
      print(f"No {object_to_find} detected in the image.")


# Adaptação Botao

In [0]:
%pip install gradio==3.1.0

In [0]:
import gradio as gr
from PIL import Image
import torch
from transformers import OwlViTProcessor, OwlViTForObjectDetection
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import io
from IPython.display import display, clear_output
import ipywidgets as widgets

# Load pre-trained model and processor once to avoid reloading them every time
processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32", clean_up_tokenization_spaces=True)
model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32")

def detect_object(image, object_to_find, threshold=0.3):
    # Process the image
    texts = [[f"a photo of a {object_to_find}"]]
    inputs = processor(text=texts, images=image, return_tensors="pt")
    outputs = model(**inputs)

    # Process the output
    target_sizes = torch.Tensor([image.size[::-1]])
    results = processor.post_process_object_detection(outputs=outputs, threshold=threshold, target_sizes=target_sizes)

    # Extract results for the first (and only) image
    boxes, scores, labels = results[0]["boxes"], results[0]["scores"], results[0]["labels"]

    # Prepare lists to store all detections
    all_coordinates = []
    all_confidences = []

    # Process all detections
    for box, score in zip(boxes, scores):
        coordinates = [round(i, 2) for i in box.tolist()]
        confidence = round(score.item(), 3)
        all_coordinates.append(coordinates)
        all_confidences.append(confidence)

    # Return all results
    return all_coordinates, all_confidences

def display_detection(image, object_name, all_coordinates, all_confidences):
    fig, ax = plt.subplots(1, figsize=(12, 8))
    
    # Display the image
    ax.imshow(image)

    # Define a color map for different confidence levels
    colors = ['r', 'g', 'b', 'y', 'c', 'm']

    # Create Rectangle patches for each detection
    for coordinates, confidence in zip(all_coordinates, all_confidences):
        color_index = min(int(confidence * len(colors)), len(colors) - 1)
        color = colors[color_index]
        
        # Unpack coordinates
        x1, y1, x2, y2 = coordinates
        
        rect = patches.Rectangle((x1, y1), 
                                 x2 - x1, 
                                 y2 - y1, 
                                 linewidth=2, edgecolor=color, facecolor='none')
        
        # Add the patch to the Axes
        ax.add_patch(rect)

        # Add label
        plt.text(x1, y1 - 10, 
                 f"{object_name} ({confidence:.3f})", 
                 color=color, fontsize=10, weight='bold')

    # Turn off axis
    plt.axis('off')
    
    # Set the title
    plt.title(f"Detected {object_name}s", fontsize=14, fontweight='bold')
    
    # Display the plot
    display(fig)
    plt.close(fig)

def process_image(image, object_to_find):
    if object_to_find == "Pacote":
        threshold = 0.1
        object_name = "package"
    else:
        threshold = 0.033
        object_name = "signature"
    
    all_coordinates, all_confidences = detect_object(image, object_name, threshold)
    
    if all_coordinates:
        display_detection(image, object_name, all_coordinates, all_confidences)
        return f"Detected {len(all_coordinates)} {object_name}(s)"
    else:
        return f"No {object_name} detected."

# Create widgets
file_upload = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload Image'
)

object_type = widgets.RadioButtons(
    options=['Pacote', 'Assinatura'],
    description='Busca:',
    disabled=False
)

detect_button = widgets.Button(
    description='Processar Elemento',
    disabled=False,
    button_style='', 
    tooltip='Click to detect objects'
)

output = widgets.Output()

# Create a loading spinner
spinner = widgets.IntProgress(
    value=0,
    min=0,
    max=10,
    description='Loading:',
    style={'bar_color': 'lightblue'},
    orientation='horizontal'
)
spinner.layout.visibility = 'hidden'

# Define button click behavior
def on_button_clicked(b):
    with output:
        clear_output()
        if not file_upload.value:
            print("Please upload an image first.")
            return
        
        # Show the spinner
        spinner.layout.visibility = 'visible'
        
        # Get the uploaded file
        uploaded_file = next(iter(file_upload.value.values()))
        image = Image.open(io.BytesIO(uploaded_file['content']))
        
        # Process the image
        result = process_image(image, object_type.value)
        print(result)
        
        # Hide the spinner
        spinner.layout.visibility = 'hidden'
        
        # Clear the file upload widget
        file_upload.value.clear()
        file_upload._counter = 0

detect_button.on_click(on_button_clicked)

# Display widgets
display(file_upload, object_type, detect_button, spinner, output)